# Build Custom HNSW Index & Ground Truth (Colab Version)

This notebook is designed to run on Google Colab (preferably with a **T4 GPU**). It will:
1. Compile your custom `adaptive_hnsw_cpp` C++ bindings on Linux.
2. Build the HNSW index using your custom logic across 10 million vectors.
3. Use the GPU to calculate the exact Ground Truth in seconds (bypassing hours of CPU time).
4. Save both the `.bin` index and `.npy` ground truth directly to your Google Drive.

**Portability Note:** The `.bin` index file created by HNSWlib in Linux (Colab) is 100% binary compatible with Windows because both use x86_64 little-endian architecture! Your teammates can simply download the file and load it locally.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Setup Working Directory & Compile Bindings
# IMPORTANT: Change this path to wherever you uploaded your code (bindings.cpp, setup.py, hnswlib/) on Drive.
WORKSPACE_DIR = "/content/drive/MyDrive/Internship/hybridAdaEf/" 

import os
os.chdir(WORKSPACE_DIR)
print(f"Working in: {os.getcwd()}")

# Compile the C++ bindings
!pip install pybind11
!python setup.py build_ext --inplace
print("Compilation successful! Library is ready.")

In [ ]:
# 3. Load the Dataset
import h5py
import numpy as np
import time
import adaptive_hnsw_cpp  # Your custom compiled bindings!

dataset_path = "/content/drive/MyDrive/deep-image-96-angular.hdf5"
print("Loading dataset...")
with h5py.File(dataset_path, 'r') as f:
    base_data = f['train'][:]  # Load all 10M into RAM
    query_data = f['test'][:]  # Load 10K queries
    print(f"Loaded {base_data.shape[0]} vectors of dimension {base_data.shape[1]}.")

In [ ]:
# 4. Build Custom HNSW Index
num_elements, dim = base_data.shape
M = 16
ef_construction = 500

print("Initializing Custom Index...")
# Assuming 'cosine' space (DeepImage uses angular/cosine)
idx = adaptive_hnsw_cpp.Index("cosine", dim)
idx.init_index(max_elements=num_elements, M=M, ef_construction=ef_construction)

print("Adding vectors to index (this will take 15-30 mins using Colab CPUs)...")
start_time = time.time()
# Note: Colab has limited vCPUs, so we batch it.
batch_size = 100000
for i in range(0, num_elements, batch_size):
    end_idx = min(i + batch_size, num_elements)
    idx.add_items(base_data[i:end_idx], np.arange(i, end_idx))
    if (i // batch_size) % 10 == 0:
        print(f"Inserted {end_idx}/{num_elements} vectors...")

print(f"Index built in {time.time() - start_time:.2f} seconds.")

# Save the index to Google Drive!
index_save_path = "/content/drive/MyDrive/deepimage_custom_hnsw.bin"
idx.save_index(index_save_path)
print(f"Index safely saved to {index_save_path}")

In [ ]:
# 5. Compute Exact Ground Truth using T4 GPU (Blazing Fast!)
!pip install faiss-gpu
import faiss

print("\nComputing exact Top-100 Ground Truth using GPU...")
res = faiss.StandardGpuResources()

# For cosine distance, we normalize vectors and use Inner Product
faiss.normalize_L2(base_data)
faiss.normalize_L2(query_data)

index_flat = faiss.IndexFlatIP(dim)
gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index_flat)

print("Adding to FAISS GPU Index...")
gpu_index_flat.add(base_data)

print("Searching queries...")
k = 100
start_time = time.time()
D, ground_truth_labels = gpu_index_flat.search(query_data, k)
print(f"Ground Truth computed in {time.time() - start_time:.2f} seconds!")

# Save Ground Truth to Google Drive
gt_save_path = "/content/drive/MyDrive/deepimage_gt_labels.npy"
np.save(gt_save_path, ground_truth_labels)
print(f"Ground Truth labels saved to {gt_save_path}")

print("\n✅ ALL DONE! Your teammates can now download the .bin and .npy files directly from Drive.")